# Lab 5.4 &mdash; Human-in-the-Loop as an Orchestration Mechanism

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Stop a compiled graph before the irreversible node with <code>interrupt_before</code>
- Resume it, recording an identity rather than a boolean
- Time out, and climb an escalation ladder that actually terminates
- Rewind into the gate and find out who the interrupt really belongs to

> **How this lab works.** You write real LangGraph code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a compiled `StateGraph`, a declared reducer, a checkpointed interrupt), so they are
> deterministic and never depend on the model. Cells marked **Run it for real** put your work
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **Module 3's checkpointing, applied.** You can only pause a run whose state you can
> write down and pick up again &mdash; an approval gate is that mechanism with a person in it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def _is_todo(exc: BaseException) -> bool:
    """Is this exception really an unfilled blank?

    LangGraph runs your nodes inside tasks, so the NameError from an unfilled BLANK can
    arrive wrapped. Walk the cause chain before calling anything a failure -- telling you
    your answer is wrong when you have not written one yet is the worst thing a lab does.
    """
    seen = set()
    while exc is not None and id(exc) not in seen:
        if isinstance(exc, NameError):
            return True
        seen.add(id(exc))
        exc = exc.__cause__ or exc.__context__
    return False

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except Exception as exc:
        if _is_todo(exc):
            print(f"[TODO] {name}")
            _results.append(None)
            return
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except Exception as exc:
        if not _is_todo(exc):
            raise
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because the "Run it for real" cells make many small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 5 labs -- the same payment exceptions, now worked
# by several agents at once, and finally priced against the single agent from Day 1.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the specialists, as LangGraph nodes
# Each one takes the graph state and returns a PARTIAL state -- exactly the node shape from
# Module 3 -- and reports what it spent. They are deterministic, so a graph's structure AND
# its cost can be asserted offline and exactly. The "Run it for real" cells put the sandbox
# model behind the same interface.

SANCTIONS_WATCH = {"NORTHWIND"}

COST = {"supervisor": 120, "ledger": 380, "policy": 420, "sanctions": 90, "writer": 610}


def agent_ledger(state: dict) -> dict:
    """Read the payment named in the state."""
    ref = state.get("ref")
    record = LEDGER.get(ref)
    if record is None:
        return {"problems": [f"no payment on file with reference {ref!r}"],
                "tokens": COST["ledger"]}
    return {"facts": {"ref": ref, **record},
            "findings": [{"by": "ledger", "source": "ledger",
                          "claim": f"{ref} is {record['status']} "
                                   f"for {record['amount']:,.2f} {record['ccy']}"}],
            "tokens": COST["ledger"]}


def agent_policy(state: dict) -> dict:
    """Say what the operating policy is for whatever went wrong."""
    code = (state.get("facts") or {}).get("reason_code")
    if code is None:
        return {"problems": ["policy ran before the reason code existed"],
                "tokens": COST["policy"]}
    return {"findings": [{"by": "policy", "source": "policy",
                          "claim": POLICY.get(code, f"no policy on file for {code}")}],
            "needs_human": code in NEEDS_HUMAN,
            "tokens": COST["policy"]}


def agent_sanctions(state: dict) -> dict:
    """A set-membership test. No model needed, and none used -- note the cost column."""
    counterparty = (state.get("facts") or {}).get("counterparty")
    listed = counterparty in SANCTIONS_WATCH
    return {"findings": [{"by": "sanctions", "source": "watchlist",
                          "claim": f"{counterparty} is "
                                   f"{'ON the watchlist' if listed else 'not on the watchlist'}"}],
            "blocked": listed,
            "tokens": COST["sanctions"]}


def agent_writer(state: dict) -> dict:
    """Turn whatever findings arrived into one recommendation."""
    findings = state.get("findings") or []
    if (state.get("facts") or {}).get("status") == "settled":
        action = "no action"                      # nothing to release; it already went
    elif state.get("blocked") or state.get("needs_human"):
        action = "hold for a human"
    else:
        action = "release"
    return {"recommendation": action,
            "rationale": [f["claim"] for f in findings],
            "tokens": COST["writer"]}


AGENTS = {"ledger": agent_ledger, "policy": agent_policy,
          "sanctions": agent_sanctions, "writer": agent_writer}
print("specialists:", ", ".join(AGENTS))

## Concept

Human-in-the-loop appears twice in this course. Here it is an **orchestration mechanism**: a way
to pause a graph, ask, and carry on. In Module 8 the same machinery is a **safety control**.

Four parts, and the one people leave out is the third:

| | |
|---|---|
| **interrupt** | `compile(interrupt_before=[...])` &mdash; stop before a named node, state saved |
| **approve** | `update_state` who said yes, then `invoke(None, cfg)` to carry on |
| **timeout** | a gate with no deadline is a run that waits until Monday |
| **escalate** | expiry is not refusal and not approval &mdash; it is a different queue |

One thing that is **not** true: &ldquo;an approval gate needs a checkpointer&rdquo;. A gate that
simply refuses to act without a named approver needs nothing at all &mdash; you will build one in
Section 1's `node_release`. What needs a checkpointer is **pause and resume**: stopping now and
finishing later, from another process, after the person replies.

## Section 1 &mdash; Interrupt before the irreversible node

The graph from Lab 5.2, with one more node on the end that actually changes something. The
checkpointer writes the state after every node; `interrupt_before` says where not to walk past.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

RELEASED = set()          # the irreversible side effect, so we can prove whether it happened

GATED_NODES = ["release"]  # nodes that may not run unattended


class GateState(TypedDict):
    ref: str
    facts: dict | None
    findings: Annotated[list, add]
    problems: Annotated[list, add]
    tokens: Annotated[int, add]
    blocked: bool
    needs_human: bool
    recommendation: str | None
    rationale: list
    approved_by: str | None
    released: bool


def node_release(state: GateState) -> dict:
    """The one node that changes the world. It refuses unless a PERSON is named.

    This much needs no checkpointer: it is a gate because the tool will not fire without
    an approver in state. The checkpointer is what lets the person answer tomorrow.
    """
    who = state.get("approved_by")
    if not isinstance(who, str) or not who.strip():
        return {"released": False, "tokens": 10,
                "problems": ["release refused: no named human approver"]}
    RELEASED.add(state["ref"])
    return {"released": True, "tokens": 40}


def node_release_unattended(state: GateState) -> dict:
    """The same action with the review moved to the END of the run. It just goes."""
    RELEASED.add(state["ref"])
    return {"released": True, "tokens": 40}


def build(checkpointer=None, gate: bool = True):
    g = StateGraph(GateState)
    g.add_node("read", agent_ledger)
    g.add_node("policy", agent_policy)
    g.add_node("screen", agent_sanctions)
    g.add_node("recommend", agent_writer)
    g.add_node("release", node_release if gate else node_release_unattended)
    g.add_edge(START, "read")
    g.add_edge("read", "policy")
    g.add_edge("read", "screen")
    g.add_edge("policy", "recommend")
    g.add_edge("screen", "recommend")
    g.add_edge("recommend", "release")
    g.add_edge("release", END)
    # An approval gate is a place the graph is not allowed to walk past on its own.
    return g.compile(checkpointer=checkpointer,
                     interrupt_before=GATED_NODES if gate else [])


def cfg(thread_id: str) -> dict:
    """A thread is one case. Two cases must never share one."""
    return {"configurable": {"thread_id": thread_id}}


def fresh_gate(ref: str) -> dict:
    return {"ref": ref, "facts": None, "findings": [], "problems": [], "tokens": 0,
            "blocked": False, "needs_human": False, "recommendation": None,
            "rationale": [], "approved_by": None, "released": False}


def pending(app, thread: str):
    """What is this thread waiting to do?"""
    return app.get_state(cfg(thread)).next

In [ ]:
# --- Self-check: Section 1   (a real checkpointed graph hitting a real interrupt -- no model)
def started(thread: str = "g1", ref: str = "PMT-1005"):
    """A fresh saver, a fresh thread, run until it stops."""
    RELEASED.clear()
    app = build(checkpointer=InMemorySaver())
    app.invoke(fresh_gate(ref), cfg(thread))
    return app

check("the run stops instead of finishing",
      lambda: pending(started(), "g1") == ("release",))
check("NOTHING WAS RELEASED",
      lambda: (started(), "PMT-1005" not in RELEASED)[1] is True,
      "the point of interrupting BEFORE the node rather than after it")
check("everything before the gate did run",
      lambda: len(started().get_state(cfg("g1")).values["findings"]) == 3)
check("so the human is looking at evidence, not at a blank form",
      lambda: started().get_state(cfg("g1")).values["recommendation"] == "hold for a human")
check("the checkpointer wrote a checkpoint per step, not one at the end",
      lambda: len(list(started().get_state_history(cfg("g1")))) >= 4,
      "that is what makes resuming, rewinding and auditing possible at all")
def _ungated():
    """The same graph compiled with no interrupt at all."""
    RELEASED.clear()
    app = build(checkpointer=InMemorySaver(), gate=False)
    app.invoke(fresh_gate("PMT-1002"), cfg("g0"))
    return app

check("an ungated build runs straight through to the end",
      lambda: pending(_ungated(), "g0") == (),
      "nothing pending means nothing stopped it -- and PMT-1002 has now been released")
check("two cases on two threads do not see each other",
      lambda: started("gA").get_state(cfg("gB")).values in ({}, None),
      "one thread per case is the whole of the isolation you get")

## Section 2 &mdash; Approval is an identity, not a boolean

Resuming is two calls. First write what the human decided into the checkpoint with
`update_state`; then `invoke(None, cfg)` &mdash; **`None` means carry on**, not start again.

`node_release` above already refuses anything that is not a name. So the only question left is
what an approval has to put into state for it to be satisfied.

In [ ]:
def resume_unapproved(app, thread: str) -> dict:
    """Carry on with nothing written. The release node decides what to do about that."""
    return app.invoke(None, cfg(thread))


def approve(app, thread: str, approved_by: str) -> dict:
    """Record WHO approved, then carry on.

    'approved: true' answers none of the questions an auditor asks -- who, and on what
    evidence. So the caller must hand over a name, and the name goes into the state that
    the release node reads.
    """
    if not isinstance(approved_by, str) or not approved_by.strip():
        raise ValueError("an approval needs a named human")
    app.update_state(cfg(thread), {"approved_by": approved_by})
    return app.invoke(None, cfg(thread))

In [ ]:
# --- Self-check: Section 2   (real update_state, real resume -- no model)
def _unapproved():
    return resume_unapproved(started("s2a"), "s2a")

def _approved(who="ops-duty-manager"):
    app = started("s2b")
    return app, approve(app, "s2b", who)

def _refused_with(who):
    """Approving with something that is not a name must leave the world unchanged."""
    app = started("s2c")
    try:
        approve(app, "s2c", who)
    except NameError:
        raise
    except ValueError:
        pass
    return "PMT-1005" not in RELEASED

check("resuming with nothing written does not release",
      lambda: _unapproved()["released"] is False)
check("...and it says why, in state a person can read",
      lambda: any("no named human approver" in p for p in _unapproved()["problems"]))
check("a bare True is not an approver",
      lambda: _refused_with(True) is True,
      "'approved: true' cannot answer 'who approved this, and on what evidence?'")
check("nor is an empty string",
      lambda: _refused_with("   ") is True)
check("a named human opens the gate",
      lambda: _approved()[1]["released"] is True)
check("and the payment actually went",
      lambda: (_approved(), "PMT-1005" in RELEASED)[1] is True)
check("the approver's name is on the final state, not just a flag",
      lambda: _approved()[1]["approved_by"] == "ops-duty-manager")
check("after resuming there is nothing pending",
      lambda: pending(_approved()[0], "s2b") == ())
check("the work done before the pause was kept, not redone",
      lambda: _approved()[1]["tokens"]
              == COST["ledger"] + COST["policy"] + COST["sanctions"] + COST["writer"] + 40,
      "a pause is not a rollback -- the checkpoint carried the findings and the bill across")

## Section 3 &mdash; Timeout, and a ladder that ends

A gate with no deadline is a run that waits for someone who has gone home. The ladder is given;
the decision is what the deadline passing actually *means*.

In [ ]:
ESCALATION = ["ops-duty-manager", "treasury-lead", "head-of-operations"]

def escalate(current, ladder=None):
    """Who to ask next. None means the ladder is exhausted and a person must own it manually."""
    ladder = ESCALATION if ladder is None else ladder
    if current is None:
        return ladder[0]
    if current not in ladder:
        return None
    i = ladder.index(current)
    return ladder[i + 1] if i + 1 < len(ladder) else None


def gate_status(waited_s: int, deadline_s: int, approver=None) -> str:
    """What to do with a gate that has been waiting."""
    if isinstance(approver, str) and approver.strip():
        return "approved"
    if waited_s < deadline_s:
        return "waiting"
    # The deadline passed and nobody answered. That is not a yes, and it is not a no --
    # it is a different queue, and someone further up owns it now.
    return "escalate"

In [ ]:
# --- Self-check: Section 3
check("an unopened gate starts at the bottom of the ladder",
      lambda: escalate(None) == "ops-duty-manager")
check("and climbs one rung at a time",
      lambda: escalate("ops-duty-manager") == "treasury-lead")
check("the ladder TERMINATES",
      lambda: escalate("head-of-operations") is None,
      "an escalation path that loops is a gate that never resolves")
check("someone outside the ladder cannot be escalated from",
      lambda: escalate("a-passing-colleague") is None)
check("inside the deadline the gate simply waits",
      lambda: gate_status(waited_s=30, deadline_s=900) == "waiting")
check("an approval short-circuits the deadline entirely",
      lambda: gate_status(waited_s=99999, deadline_s=900, approver="treasury-lead") == "approved")
check("expiry is neither approval nor refusal",
      lambda: gate_status(waited_s=901, deadline_s=900) not in ("approved", "refused"),
      "a timeout that auto-approves is not a gate; one that auto-refuses loses real work")
check("and what it is instead is something the ladder can act on",
      lambda: gate_status(waited_s=901, deadline_s=900) == "escalate")

def _ladder():
    who, waited = None, 0
    while True:
        who = escalate(who)
        if who is None:
            print("  ladder exhausted -- this case now belongs to a person, not to the graph")
            break
        waited += 900
        print(f"  after {waited // 60:>3} min -> ask {who}  ({gate_status(waited, 900)})")
guard(_ladder)

## Section 4 &mdash; Placement, and who the interrupt belongs to

Two questions decide whether you built a gate or a notification.

**Where is it?** At the moment the human says no, has anything irreversible already happened?

**Whose is it?** Rewind to an earlier checkpoint and replay. A run-scoped interrupt would sail
through; a graph-scoped one stops again. Find out which you have &mdash; this catches people out.

In [ ]:
def no_is_free(gate: bool, ref: str = "PMT-1005") -> bool:
    """Run it, have nobody approve, and ask whether anything happened anyway."""
    RELEASED.clear()
    app = build(checkpointer=InMemorySaver(), gate=gate)
    app.invoke(fresh_gate(ref), cfg("placement"))
    return ref not in RELEASED


def checkpoint_before(app, thread: str, node: str):
    """The config of the checkpoint at which `node` was the next thing to run."""
    for snap in app.get_state_history(cfg(thread)):
        if snap.next == (node,):
            return snap.config      # a config carrying that checkpoint_id, not just the thread
    return None


def rewind_and_replay(thread: str = "rw"):
    """Approve once, then go back to before the recommendation and run it again."""
    app = started(thread)
    approve(app, thread, "ops-duty-manager")            # it completed, once
    back = checkpoint_before(app, thread, "recommend")
    app.invoke(None, back)                              # replay from the older checkpoint
    return app

In [ ]:
# --- Self-check: Section 4
check("with the gate before the write, saying nothing costs nothing",
      lambda: no_is_free(gate=True) is True)
check("with the review after the write, the payment already went",
      lambda: no_is_free(gate=False) is False,
      "the reviewer sees a complete, sourced summary of something they can no longer stop")
check("both runs showed the reviewer exactly the same evidence",
      lambda: len(started("cmp").get_state(cfg("cmp")).values["findings"]) == 3,
      "quality of evidence was never the difference -- placement was")
check("checkpoint_before finds a real point in the past",
      lambda: checkpoint_before(started("cb"), "cb", "recommend") is not None)
check("what it returns addresses a checkpoint, not just the thread",
      lambda: "checkpoint_id" in checkpoint_before(started("cb2"), "cb2",
                                                   "recommend")["configurable"],
      "a config with only a thread_id points at NOW, which is not a rewind")
check("rewinding into a gated graph PAUSES AT THE GATE AGAIN",
      lambda: pending(rewind_and_replay("rw1"), "rw1") == ("release",),
      "the interrupt belongs to the compiled graph, not to a run -- every path through it "
      "pauses, including a replay of one that was already approved")
check("and the earlier approval did not come back with the rewind",
      lambda: rewind_and_replay("rw2").get_state(cfg("rw2")).values["approved_by"] is None,
      "you rewound to a checkpoint that predates the approval, so the person decides again")

def _placement():
    for label, gate in (("before the write", True), ("after the write ", False)):
        free = no_is_free(gate=gate)
        print(f"  gate {label}:  'no' still free? {'yes -- a gate' if free else 'NO -- a notification'}")
    app = rewind_and_replay("rw3")
    print(f"\n  after the rewind the thread is pending: {pending(app, 'rw3')}")
guard(_placement)

## Run it for real

Render the checkpoint the way a human reviewer would see it and ask the model to write the
approval request. What you are judging is whether the state you checkpointed contains enough for
a person to say no.

In [ ]:
if llm_ready():
    def _brief():
        app = started("brief")
        values = app.get_state(cfg("brief")).values
        evidence = "\n".join(f"- [{f['by']}, source={f['source']}] {f['claim']}"
                             for f in values["findings"])
        reply = ask("Write a short approval request for a duty manager. State what is being asked, "
                    "the evidence for and against, and what happens if they do nothing.\n\n"
                    f"Action awaiting approval: {pending(app, 'brief')} {values['ref']}\n"
                    f"Agent recommendation: {values.get('recommendation')}\n"
                    f"Findings:\n{evidence}")
        print(reply.strip()[:600])
    guard(_brief)

### Read it

If the model has to hedge or invent, your checkpoint is missing something a reviewer needs &mdash;
and that is a state design problem, not a prompt problem. A good approval request is mostly a
rendering of state you already had.

**Section 4 is the one to look at twice.** The replay ran forward and stopped at the gate again,
even though that thread had already been approved once. The interrupt is a property of the
**compiled graph**, not of a run: every path through it pauses. People who assume otherwise build
a &ldquo;replay for audit&rdquo; feature and are surprised to find it asking for approvals.

**What you take from this lab:** interrupt before the node, not after it; record an identity
rather than a boolean; give every gate a deadline and a ladder that ends; and remember that the
refusal in `node_release` needed no checkpointer at all &mdash; the checkpointer bought you *later*,
not *safer*.

In [ ]:
score()

## Your turn

1. Swap `InMemorySaver` for `SqliteSaver` pointed at a file under `WORK`. Run to the gate,
   restart the kernel, rebuild against the same file and resume. That is recovery after a crash,
   and it is why in-memory is a development convenience only.
2. Gate on a **condition** rather than always: pause only when `needs_human` is true.
   `interrupt_before` is static, so this belongs in a conditional edge to a node that interrupts.
   Confirm PMT-1002 runs straight through.
3. `approve` trusts its caller for the name. Where does that name actually have to come from for
   the audit trail to mean anything, and what stops an agent from supplying it?